# TruFor:Leveraging all-round clues for trustworthy image forgery detection and localization (CVPR 2023)

# 背景

## 场景介绍
针对篡改图片的篡改位置定位和真伪性检测

## 之前方法的不足
没有篡改定位与真伪性检测任务结合；提取的特征鲁棒性不足

## 本文方法
这篇文章提出TruFor检测框架——一种可广泛应用于各类图像篡改方法的取证系统，其检测范围涵盖传统廉价伪造（cheapfakes）到基于深度学习的新型篡改技术。该框架通过基于Transformer的融合架构，同时提取RGB图像与习得的噪声敏感指纹中的高层及底层痕迹。其中噪声敏感指纹仅通过自监督方式在真实数据上训练，能够内化相机内外处理流程相关的伪影特征。通过识别待测图像与原始图像固有规律模式之间的偏差来实现篡改检测，这种异常检测机制使本方法能稳健识别各类局部篡改，确保泛化能力。除提供像素级定位图与整图完整性评分外，本方法还会输出可靠性热力图，用以标记定位预测可能存在误差的区域，这一特性对于减少法证应用中的误报及实现大规模分析尤为重要。

# 训练

## 预训练权重
预训练的 Noiseprint++ 和 SegFormer-B2 权重已经包含在`pretrained_models` 文件夹中。

## 训练数据集
如果希望增加训练数据集，请在 `project_config.py` 文件中更新数据集的路径。
（如果希望直接使用原文数据集，则该部分可以跳过）

原文论文中使用的数据集路径：
- tampCOCO 和 compRAISE: https://github.com/mjkwon2021/CAT-Net
- FantasticReality: CAT-Net 作者在此处提供了链接 https://github.com/mjkwon2021/CAT-Net/issues/51
- CASIA 2.0 revised: https://github.com/namtpham/casia2groundtruth
- IMD: https://staff.utia.cas.cz/novozada/db/IMD2020.zip

要添加您自己的数据集：
- 在 `dataset` 文件夹中创建一个 dataloader（您可以使用现有的作为参考）
- 将其添加到 `data_core.py` 文件中（在 `mode == "train"` 和 `mode == "valid"` 两个部分都要添加）
- 要使用该数据集，请将其添加到配置文件的 `DATASET.TRAIN` 和/或 `DATASET.VALID` 选项的列表中

## 命令行参数与输出

命令行参数：
- `-g` 或 `--gpu`: 默认为 GPU '0'。如果您想使用 CPU，请设置为 '-1'。您可以在同一设备上使用多个 GPU 运行（例如 `-g 0 1`）。
- `-exp` 或 `--experiment`: 实验的名称。它必须与配置文件（不带扩展名）同名。

任何其他配置选项（用于在不编辑 .yaml 文件的情况下更改值）都必须以 `参数名称 参数值` 的形式放在命令的末尾，使用配置文件中包含的参数名称。
例如，要在训练开始前执行一个额外的验证步骤，您可以在命令末尾添加`VALID.FIRST_VALID True`。要更改批量大小（batch size），请在 `TRAIN.BATCH_SIZE_PER_GPU` 设置中进行修改。

## 使用提供的配置进行训练（复现论文结果）
### 阶段 1：训练定位网络

```
python train.py -exp trufor_ph2
```

### 阶段 2：训练检测网络和置信度估计器

首先，请确保 `lib/config/trufor_ph3.yaml` 中的 `TRAIN.PRETRAINING` 包含阶段1权重的路径。然后运行：

```
python train.py -exp trufor_ph3
```

您也可以直接在命令中指定，而无需编辑 yaml 文件：

```
python train.py -exp trufor_ph3 TRAIN.PRETRAINING "weights/trufor_ph2/best.pth.tar"
```

## 自定义训练

如果您想创建自己的训练，请在 `lib/config` 文件夹中复制 `trufor_ph2.yaml` 和 `trufor_ph3.yaml` 文件，然后根据您的需求重命名和编辑它们。
然后，遵循与上述相同的训练说明，在 `-exp` 中使用您的配置文件的名称。

# 推理

## 命令行参数与输出

命令行参数：
- `-g` 或 `--gpu`: 默认为 GPU '0'。如果您想使用 CPU，请设置为 '-1'。
- `-in` 或 `--input`: 默认为 "images/"。可以是一个文件、一个目录或一个 glob 表达式。
- `-out` 或 `--output`: 输出文件夹。
- `-exp` 或 `--experiment`: 实验的名称。它必须与配置文件（不带扩展名）同名。
- `--save_np`: 如果您想同时保存 Noiseprint++。

任何其他配置选项（用于在不编辑 .yaml 文件的情况下更改值）都必须以 `参数名称 参数值` 的形式放在命令的末尾，使用配置文件中包含的参数名称。
例如，`TEST.MODEL_FILE "pretrained_models/trufor.pth.tar"`

输出是一个 .npz 文件，包含以下内容：
- **'map'**: 异常定位图
- **'conf'**: 置信度图
- **'score'**: 分数，范围在 [0,1]
- **'np++'**: Noiseprint++（如果指定了 `--save_np` 标志参数）
- **'imgsize'**: 图像尺寸

## 预训练的权重

下载[权重文件](https://www.grip.unina.it/download/prog/TruFor/TruFor_weights.zip)并将其解压到 "pretrained_models" 文件夹（已经处理好）。
MD5 值为 7bee48f3476c75616c3c5721ab256ff8。

然后运行：
```
python test.py -in /workspace/user-data/models/TruFor/dataset/CASIAv1/tamper/Sp_S_NRN_R_art0056_art0056_0086.jpg -out ./output -exp trufor_ph3 TEST.MODEL_FILE "pretrained_models/trufor.pth.tar" 
```

## 使用您自己训练的权重进行推理
`TEST.MODEL_FILE` 选项不是必需的，因为它会使用在 `-exp` 中指定的名称。
```
python test.py -in /workspace/user-data/models/TruFor/dataset/CASIAv1/tamper -out ./output -exp name_of_your_yaml_ph2
```

# 评估指标

在 `metrics.py` 文件中，您可以找到我们用来计算评估指标的函数。<br/>
定位指标只能在伪造图像上计算，并且真实标签（ground truth）**必须将原始像素设为 0，伪造像素设为 1**。<br/>
在计算 F1 分数时，我们取使用定位图计算的 F1 分数和使用其反向图计算的 F1 分数中的最大值。
我们不考虑真实标签中靠近伪造区域边界的像素，因为在大多数情况下它们并不准确。

# 可视化

要可视化单张图像的输出，请运行以下命令（我这里举个例子）：
```
python visualize.py --image /workspace/user-data/models/TruFor/dataset/CASIAv1/tamper/Sp_S_NRN_R_art0056_art0056_0086.jpg --output /workspace/user-data/models/TruFor/code/output/Sp_S_NRN_R_art0056_art0056_0086.jpg.npz -gt /workspace/user-data/models/TruFor/dataset/CASIAv1/gt/Sp_S_NRN_R_art0056_art0056_0086_gt.png
```
注意提供掩码（mask）是可选的。

# 读npz文件

执行read_npz.py文件，并将下面代码中路径修改为你期望的。
```
# 1. 定义你的 .npz 文件路径
# 请将 'path/to/your/file.jpg.npz' 替换为你的实际文件路径
file_path = 'output_folder/Sp_D_CND_A_pla0005_pla0023_0281.jpg.npz' 
```